In [ ]:
import h5py
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# --- 1. LOAD EMBEDDINGS FROM HDF5 ---
H5_PATH = "../embeddings/task4_dna_rna_Vir2vec-422M.h5"

with h5py.File(H5_PATH, "r") as f:
    X = np.array(f["embeddings"][:])
    raw_labels = [l.decode("utf-8") if isinstance(l, bytes) else str(l) for l in f["labels"][:]]

encoder = LabelEncoder()
y = encoder.fit_transform(raw_labels)

# --- 2. PIPELINE & HYPERPARAMETER GRID ---
pipeline = Pipeline([
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])

param_grid = {
    "clf__max_depth": [None, 10, 30],
    "clf__max_features": ["sqrt", None],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__class_weight": [None, "balanced"]
}

# --- 3. NESTED 5x3 STRATIFIED CROSS-VALIDATION ---
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42 + fold_idx)
    grid = GridSearchCV(pipeline, param_grid, scoring="balanced_accuracy", cv=inner_cv, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    y_pred = grid.best_estimator_.predict(X_test)
    
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    fold_metrics.append({
        "fold": fold_idx,
        "balanced_accuracy": bal_acc,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred)
    })
    print(f"Fold {fold_idx}: Balanced Acc = {bal_acc:.4f}")

# --- 4. SUMMARY ---
df_res = pd.DataFrame(fold_metrics)
print("\n--- Final Random Forest Results ---")
print(f"Macro Balanced Accuracy: {df_res['balanced_accuracy'].mean():.4f} ± {df_res['balanced_accuracy'].std():.4f}")